# IMDB Model

In [17]:
from wordcloud import WordCloud, STOPWORDS
import pandas as pd
import string
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer, TfidfVectorizer
from bs4 import BeautifulSoup
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import BernoulliNB
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression

In [18]:
imdb = pd.read_csv('./data/imdb.csv')
imdb.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [19]:
def remtags(review):
    soup = BeautifulSoup(review)
    return soup.get_text()

In [20]:
def remove_stopwords(review):
    words = review.split()
    words = [word for word in words if word.lower() not in STOPWORDS]
    return " ".join(words)

In [21]:
imdb['review'] = imdb['review'].apply(remtags).str.lower()

In [22]:
imdb['review'] = imdb['review'].str.replace(r'[^A-Za-z0-9]',' ',regex=True)

In [23]:
imdb['review'] = imdb['review'].apply(remove_stopwords)

In [24]:
imdb

,review,sentiment
0,one reviewers mentioned watching 1 oz episode ...,positive
1,wonderful little production filming technique ...,positive
2,thought wonderful way spend time hot summer we...,positive
3,basically s family little boy jake thinks s zo...,negative
4,petter mattei s love time money visually stunn...,positive
...,...,...
40154,small john ford western stars cast character a...,positive
40155,early film flaws predictable plot overlong sce...,positive
40156,wondered didn t peggy sue got married first ca...,negative
40157,used love muppets muppet movie great muppet ca...,negative


In [25]:
x = imdb.iloc[:,0]
x

0        one reviewers mentioned watching 1 oz episode ...
1        wonderful little production filming technique ...
2        thought wonderful way spend time hot summer we...
3        basically s family little boy jake thinks s zo...
4        petter mattei s love time money visually stunn...
                               ...                        
40154    small john ford western stars cast character a...
40155    early film flaws predictable plot overlong sce...
40156    wondered didn t peggy sue got married first ca...
40157    used love muppets muppet movie great muppet ca...
40158    obviously prototype mick dundee adventures bar...
Name: review, Length: 40159, dtype: str

In [26]:
y = imdb.iloc[:,1]
y

0        positive
1        positive
2        positive
3        negative
4        positive
           ...   
40154    positive
40155    positive
40156    negative
40157    negative
40158    positive
Name: sentiment, Length: 40159, dtype: str

In [27]:
xtrain,xtest,ytrain,ytest = train_test_split(x,y,test_size=0.2,random_state=42,stratify=y)

In [28]:
vect = CountVectorizer()

In [29]:
xtrain = vect.fit_transform(xtrain)

In [30]:
xtrain[:5]

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 363 stored elements and shape (5, 85842)>

In [31]:
xtest = vect.transform(xtest)

In [32]:
bnb = BernoulliNB()

In [33]:
modelbnb = bnb.fit(xtrain,ytrain)

In [34]:
predbnb = modelbnb.predict(xtest)

In [35]:
accuracy_score(ytest,predbnb)

0.8493525896414342

In [47]:
pd.crosstab(ytest,predbnb)

col_0,negative,positive
sentiment,,
negative,3549,467
positive,743,3273


In [37]:
tfidf = TfidfTransformer()

In [38]:
tfidf1 = tfidf.fit(xtrain)

In [40]:
tftrain = tfidf1.transform(xtrain)
tftest = tfidf1.transform(xtest)

In [41]:
print(tftrain)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 3129818 stored elements and shape (32127, 85842)>
  Coords	Values
  (0, 1872)	0.03360427458867958
  (0, 3439)	0.09899792823067313
  (0, 4141)	0.038811669561911816
  (0, 4879)	0.20632722126154107
  (0, 5639)	0.056992332591853434
  (0, 7218)	0.10393963895600777
  (0, 7664)	0.2576551805224239
  (0, 7767)	0.09346846438630875
  (0, 7804)	0.11728131621898735
  (0, 8387)	0.0860620884220086
  (0, 12673)	0.10870092721522442
  (0, 12695)	0.040752414272495735
  (0, 12725)	0.07910022009412795
  (0, 15519)	0.10316361063077054
  (0, 16756)	0.08680381587852079
  (0, 16757)	0.11433834808740513
  (0, 17993)	0.14091225238695135
  (0, 18209)	0.09056167920359612
  (0, 19271)	0.06559927726002769
  (0, 20317)	0.10393963895600777
  (0, 20946)	0.06045711862867187
  (0, 21628)	0.08595945228853498
  (0, 21991)	0.09845954014715837
  (0, 22548)	0.08747457109532794
  (0, 22549)	0.08626978840889621
  :	:
  (32126, 72184)	0.03786356216148834
  (32126, 726

In [43]:
modeltf = bnb.fit(tftrain,ytrain)

In [44]:
pred2 = modeltf.predict(tftest)

In [45]:
accuracy_score(pred2,ytest)

0.8493525896414342

In [48]:
pd.crosstab(ytest,pred2)

col_0,negative,positive
sentiment,,
negative,3549,467
positive,743,3273
